# Multi-Task Learning for Trading

This notebook demonstrates how to use multi-task learning to simultaneously optimize multiple trading objectives.

## Setup

First, let's import the necessary modules and set up our environment.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from src.models.multi_task_learner import MultiTaskModel
from src.data.binance_fetcher import BinanceFetcher
from src.features.feature_engineering import FeatureEngineer

## Configuration

Load and configure multi-task learning settings.

In [ ]:
# Load configuration
with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Define task configurations
task_configs = {
    "price_prediction": {
        "output_dim": 1,
        "hidden_dims": [64, 32],
        "weight": 1.0
    },
    "volatility_prediction": {
        "output_dim": 1,
        "hidden_dims": [64, 32],
        "weight": 0.5
    },
    "regime_classification": {
        "output_dim": 3,
        "hidden_dims": [64, 32],
        "weight": 0.3
    }
}

## Data Preparation

Fetch historical data and prepare tasks for multi-task learning.

In [ ]:
# Initialize data fetcher
fetcher = BinanceFetcher(config['data']['binance'])

# Fetch historical data
data = await fetcher.fetch_historical_data()

# Initialize feature engineer
engineer = FeatureEngineer(
    input_file='data/raw/BTCUSDT_data.parquet',
    output_dir='data/processed',
    config=config
)

# Generate features
await engineer.generate_features()

## Task Generation

Create different tasks for multi-task learning.

In [ ]:
def create_tasks(data, window_size=100):
    """Create tasks for multi-task learning."""
    # Price prediction task
    returns = data['close'].pct_change()
    price_targets = returns.shift(-1)
    
    # Volatility prediction task
    volatility = returns.rolling(window=20).std()
    volatility_targets = volatility.shift(-1)
    
    # Regime classification task
    vol_quantiles = volatility.quantile([0.33, 0.66])
    regimes = pd.cut(
        volatility,
        bins=[-np.inf, vol_quantiles[0.33], vol_quantiles[0.66], np.inf],
        labels=[0, 1, 2]
    )
    
    return {
        'features': data['features'].values,
        'price_prediction': price_targets.values,
        'volatility_prediction': volatility_targets.values,
        'regime_classification': regimes.values
    }

# Create tasks
tasks_data = create_tasks(data)

# Create dataset
class MultiTaskDataset(torch.utils.data.Dataset):
    def __init__(self, tasks_data):
        self.features = torch.FloatTensor(tasks_data['features'])
        self.targets = {
            task: torch.FloatTensor(tasks_data[task])
            for task in ['price_prediction', 'volatility_prediction', 'regime_classification']
        }
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return (
            self.features[idx],
            {task: targets[idx] for task, targets in self.targets.items()}
        )

# Split data
split_idx = int(0.8 * len(tasks_data['features']))
train_data = {k: v[:split_idx] for k, v in tasks_data.items()}
val_data = {k: v[split_idx:] for k, v in tasks_data.items()}

# Create datasets
train_dataset = MultiTaskDataset(train_data)
val_dataset = MultiTaskDataset(val_data)

# Create data loaders
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=64
)

## Model Training

Train the multi-task model on all tasks.

In [ ]:
# Initialize model
model = MultiTaskModel(
    config=config,
    task_configs=task_configs,
    shared_dim=128,
    uncertainty_weighting=True
)

# Initialize optimizer
optimizer = torch.optim.Adam(model.parameters())

# Training loop
num_epochs = 50
history = {
    'train_loss': [],
    'val_loss': [],
    'task_losses': {task: [] for task in task_configs.keys()},
    'task_weights': []
}

for epoch in range(num_epochs):
    # Training
    model.train()
    train_metrics = []
    
    for batch in train_loader:
        metrics = model.train_step(batch, optimizer)
        train_metrics.append(metrics)
    
    # Validation
    model.eval()
    val_metrics = model.evaluate(val_loader)
    
    # Record history
    avg_train_metrics = pd.DataFrame(train_metrics).mean().to_dict()
    history['train_loss'].append(avg_train_metrics['total_loss'])
    history['val_loss'].append(val_metrics['total_loss'])
    
    for task in task_configs.keys():
        history['task_losses'][task].append(avg_train_metrics[task])
    
    history['task_weights'].append(model.get_task_weights())
    
    # Print progress
    print(f"Epoch {epoch+1}/{num_epochs}:")
    print(f"  Train Loss: {avg_train_metrics['total_loss']:.4f}")
    print(f"  Val Loss: {val_metrics['total_loss']:.4f}")
    
    for task, loss in val_metrics.items():
        if task != 'total_loss':
            print(f"  {task}: {loss:.4f}")

## Performance Analysis

Analyze the model's performance on different tasks.

In [ ]:
def plot_training_history(history):
    """Plot training history."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Overall loss
    axes[0,0].plot(history['train_loss'], label='Train')
    axes[0,0].plot(history['val_loss'], label='Validation')
    axes[0,0].set_title('Total Loss')
    axes[0,0].legend()
    
    # Task-specific losses
    for task, losses in history['task_losses'].items():
        axes[0,1].plot(losses, label=task)
    axes[0,1].set_title('Task Losses')
    axes[0,1].legend()
    
    # Task weights
    weights = pd.DataFrame(history['task_weights'])
    for task in weights.columns:
        axes[1,0].plot(weights[task], label=task)
    axes[1,0].set_title('Task Weights')
    axes[1,0].legend()
    
    plt.tight_layout()
    plt.show()

# Plot training history
plot_training_history(history)

## Task-Specific Analysis

Analyze performance on each task separately.

In [ ]:
def analyze_task_performance(model, data_loader):
    """Analyze performance on each task."""
    model.eval()
    predictions = []
    targets = []
    
    with torch.no_grad():
        for features, batch_targets in data_loader:
            batch_predictions = model(features)
            predictions.append(batch_predictions)
            targets.append(batch_targets)
    
    # Combine batches
    predictions = {task: torch.cat([p[task] for p in predictions])
                  for task in predictions[0].keys()}
    targets = {task: torch.cat([t[task] for t in targets])
              for task in targets[0].keys()}
    
    # Plot results
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Price prediction
    axes[0].scatter(targets['price_prediction'],
                   predictions['price_prediction'])
    axes[0].plot([-1, 1], [-1, 1], 'r--')
    axes[0].set_title('Price Prediction')
    axes[0].set_xlabel('True Returns')
    axes[0].set_ylabel('Predicted Returns')
    
    # Volatility prediction
    axes[1].scatter(targets['volatility_prediction'],
                   predictions['volatility_prediction'])
    axes[1].plot([0, 0.1], [0, 0.1], 'r--')
    axes[1].set_title('Volatility Prediction')
    axes[1].set_xlabel('True Volatility')
    axes[1].set_ylabel('Predicted Volatility')
    
    # Regime classification
    confusion = torch.zeros(3, 3)
    pred_regimes = predictions['regime_classification'].argmax(dim=1)
    true_regimes = targets['regime_classification'].long()
    
    for t, p in zip(true_regimes, pred_regimes):
        confusion[t, p] += 1
    
    axes[2].imshow(confusion)
    axes[2].set_title('Regime Classification')
    axes[2].set_xlabel('Predicted Regime')
    axes[2].set_ylabel('True Regime')
    
    plt.tight_layout()
    plt.show()

# Analyze performance
analyze_task_performance(model, val_loader)

## Save Model

Save the trained multi-task model for future use.

In [ ]:
# Save model
save_path = 'models/multi_task_model.pt'
model.save_checkpoint(save_path)
print(f"Model saved to {save_path}")